# Multi-Agent SysML Sequence Diagram Generator
Uses LangGraph + Groq (LLaMA 3.1)

Upload requirements → Generate SysML sequence diagrams

In [ ]:
!pip install langgraph langchain langchain-groq pypdf plantuml matplotlib

In [ ]:
import os
from langchain_groq import ChatGroq

# Set your Groq API Key
os.environ["GROQ_API_KEY"] = "YOUR_API_KEY_HERE"

llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0.2
)

## 📂 File Upload

In [ ]:
from google.colab import files
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
file_name

In [ ]:
from pypdf import PdfReader

def load_requirements(file_path):
    if file_path.endswith(".txt"):
        return open(file_path).read()
    elif file_path.endswith(".pdf"):
        reader = PdfReader(file_path)
        return "\n".join([p.extract_text() for p in reader.pages])
    else:
        raise ValueError("Unsupported file type")

requirements_text = load_requirements(file_name)
requirements_text[:1000]

## 🤖 Multi-Agent Definitions

In [ ]:
def extract_functionalities(text):
    prompt = f"""
    Extract distinct system functionalities from the following requirements.
    Return as a numbered list.
    
    Requirements:
    {text}
    """
    return llm.invoke(prompt).content


def extract_interactions(functionality):
    prompt = f"""
    For the given functionality, identify:
    - Actors
    - System components
    - Sequence of interactions

    Provide structured steps.

    Functionality:
    {functionality}
    """
    return llm.invoke(prompt).content


def generate_sequence_diagram(interaction):
    prompt = f"""
    Convert the following interaction into a SysML sequence diagram using PlantUML.
    
    Rules:
    - Use actors and participants
    - Include message flow
    - Use proper sequence diagram syntax

    Interaction:
    {interaction}
    """
    return llm.invoke(prompt).content


def validate_diagram(diagram):
    prompt = f"""
    Validate the following PlantUML sequence diagram.
    Fix any syntax or logical issues.

    Diagram:
    {diagram}
    """
    return llm.invoke(prompt).content

## 🔄 LangGraph Pipeline

In [ ]:
from langgraph.graph import StateGraph

class State(dict):
    pass


def functionality_node(state):
    state["functionalities"] = extract_functionalities(state["requirements"])
    return state


def interaction_node(state):
    funcs = state["functionalities"].split("\n")
    interactions = []

    for f in funcs:
        if f.strip():
            interactions.append(extract_interactions(f))

    state["interactions"] = interactions
    return state


def diagram_node(state):
    diagrams = []

    for interaction in state["interactions"]:
        diagram = generate_sequence_diagram(interaction)
        validated = validate_diagram(diagram)
        diagrams.append(validated)

    state["diagrams"] = diagrams
    return state


builder = StateGraph(State)

builder.add_node("functionality", functionality_node)
builder.add_node("interaction", interaction_node)
builder.add_node("diagram", diagram_node)

builder.set_entry_point("functionality")

builder.add_edge("functionality", "interaction")
builder.add_edge("interaction", "diagram")

graph = builder.compile()

## ▶️ Run Pipeline

In [ ]:
state = {
    "requirements": requirements_text
}

result = graph.invoke(state)

diagrams = result["diagrams"]
len(diagrams)

## 🧾 Display Diagrams

In [ ]:
for i, d in enumerate(diagrams):
    print(f"\n=== Diagram {i+1} ===\n")
    print(d)

## 🖼️ Optional: Render Diagrams

In [ ]:
from plantuml import PlantUML

plantuml = PlantUML(url="http://www.plantuml.com/plantuml/img/")

for i, diagram in enumerate(diagrams):
    with open(f"diagram_{i}.txt", "w") as f:
        f.write(diagram)

    plantuml.processes_file(f"diagram_{i}.txt")